# TASK 1 — Build Nearest-Neighbor Graph (Embedding Geometry)

**Goal**: Compute and cache nearest neighbors in embedding space once, to be reused for all descriptors.

## Steps
1. Using validation embeddings only, compute kNN using cosine distance
2. Use k ∈ {10, 50, 100}
3. For each spectrum i, store:
   - Neighbor indices
   - Cosine similarities
4. Build two versions:
   - **A) Inclusive**: allow same-molecule neighbors
   - **B) Exclusive**: filter neighbors from same molecule
5. Use sklearn NearestNeighbors for efficiency
6. Fix random seeds where applicable

## Outputs
- Cached neighbor graphs:
  - `knn_k10_inclusive.pkl`
  - `knn_k10_exclusive.pkl`
  - `knn_k50_inclusive.pkl`
  - `knn_k50_exclusive.pkl`
  - `knn_k100_inclusive.pkl`
  - `knn_k100_exclusive.pkl`
- Log neighbor stats (avg neighbors per spectrum after exclusion)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Imports complete")

## 1. Setup and Load Data

In [ ]:
# Paths
results_dir = Path('../results/indicators')
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Results directory: {results_dir}")

In [ ]:
# Load indicator_data from TASK 0
print("Loading indicator_data cache from TASK 0...")

cache_path = results_dir / 'indicator_data.pkl'

try:
    indicator_data = joblib.load(cache_path)
    embeddings = indicator_data['embeddings']
    spectrum_to_molecule = indicator_data['spectrum_to_molecule']
    molecule_to_spectra = indicator_data['molecule_to_spectra']
    print(f"  ✓ Loaded from disk: {cache_path}")
except FileNotFoundError:
    print(f"  ❌ Cache not found: {cache_path}")
    print(f"  Please run TASK 0 notebook first to generate the cache.")
    raise

print(f"\n  Spectra: {embeddings.shape[0]:,}")
print(f"  Embedding dimension: {embeddings.shape[1]}")
print(f"  Unique molecules: {len(molecule_to_spectra):,}")

## 2. Compute kNN Graphs

In [ ]:
# Normalize embeddings for cosine distance
print("Normalizing embeddings...")
embeddings_normalized = normalize(embeddings, norm='l2')

print(f"  Shape: {embeddings_normalized.shape}")
print(f"  Sample norms (should be ~1.0): {np.linalg.norm(embeddings_normalized[:5], axis=1)}")

In [ ]:
# Compute kNN for k=10, 50, 100
print("\nComputing kNN graphs using cosine distance...")
print("="*80)

k_values = [10, 50, 100]
knn_graphs_inclusive = {}  # k -> graph
knn_graphs_exclusive = {}   # k -> graph

for k in k_values:
    print(f"\nk = {k}")
    
    # Compute kNN with k+1 to include self, then remove self
    print(f"  Fitting NearestNeighbors (k={k}+1)...")
    nbrs = NearestNeighbors(n_neighbors=k+1, metric='cosine', n_jobs=-1)
    nbrs.fit(embeddings_normalized)
    
    distances, indices = nbrs.kneighbors(embeddings_normalized)
    
    # Remove self (index 0)
    distances = distances[:, 1:]  # (n_spectra, k)
    indices = indices[:, 1:]       # (n_spectra, k)
    
    # Convert distances to similarities (1 - distance for cosine)
    similarities = 1 - distances
    
    print(f"  Distances shape: {distances.shape}")
    print(f"  Distance stats: min={distances.min():.4f}, max={distances.max():.4f}, mean={distances.mean():.4f}")
    print(f"  Similarity stats: min={similarities.min():.4f}, max={similarities.max():.4f}, mean={similarities.mean():.4f}")
    
    # Store INCLUSIVE version
    knn_graphs_inclusive[k] = {
        'indices': indices,
        'distances': distances,
        'similarities': similarities,
        'k': k,
        'type': 'inclusive'
    }
    print(f"  ✓ Inclusive: stored")

print(f"\n{'='*80}")
print(f"✓ All kNN graphs computed")

## 3. Build Exclusive (Same-Molecule Filtered) Versions

In [ ]:
print("Building exclusive (same-molecule filtered) versions...")
print("="*80)

for k in k_values:
    print(f"\nk = {k}")
    
    indices_incl = knn_graphs_inclusive[k]['indices']
    distances_incl = knn_graphs_inclusive[k]['distances']
    similarities_incl = knn_graphs_inclusive[k]['similarities']
    
    # Create exclusive version by filtering same-molecule neighbors
    n_spectra = embeddings.shape[0]
    indices_excl = []
    distances_excl = []
    similarities_excl = []
    
    neighbors_kept_per_spectrum = []
    
    for i in range(n_spectra):
        # Get inclusive neighbors
        neighbors = indices_incl[i]
        dists = distances_incl[i]
        sims = similarities_incl[i]
        
        # Filter: keep only neighbors from different molecules
        same_mol = spectrum_to_molecule[neighbors] == spectrum_to_molecule[i]
        mask = ~same_mol
        
        neighbors_filtered = neighbors[mask]
        dists_filtered = dists[mask]
        sims_filtered = sims[mask]
        
        # If we filtered out too many, keep only first k different-molecule neighbors
        # (This is rare if k is large relative to spectra per molecule)
        n_kept = len(neighbors_filtered)
        neighbors_kept_per_spectrum.append(n_kept)
        
        # Pad or truncate to maintain uniform shape (use -1 for padding)
        if n_kept < k:
            # Pad with -1 (invalid index)
            neighbors_filtered = np.pad(neighbors_filtered, (0, k - n_kept), constant_values=-1)
            dists_filtered = np.pad(dists_filtered, (0, k - n_kept), constant_values=np.nan)
            sims_filtered = np.pad(sims_filtered, (0, k - n_kept), constant_values=np.nan)
        
        indices_excl.append(neighbors_filtered[:k])
        distances_excl.append(dists_filtered[:k])
        similarities_excl.append(sims_filtered[:k])
    
    indices_excl = np.array(indices_excl)
    distances_excl = np.array(distances_excl)
    similarities_excl = np.array(similarities_excl)
    
    # Store EXCLUSIVE version
    knn_graphs_exclusive[k] = {
        'indices': indices_excl,
        'distances': distances_excl,
        'similarities': similarities_excl,
        'k': k,
        'type': 'exclusive'
    }
    
    # Statistics
    avg_neighbors = np.mean(neighbors_kept_per_spectrum)
    min_neighbors = np.min(neighbors_kept_per_spectrum)
    max_neighbors = np.max(neighbors_kept_per_spectrum)
    
    print(f"  ✓ Exclusive: stored")
    print(f"    Avg different-molecule neighbors: {avg_neighbors:.1f}")
    print(f"    Range: [{min_neighbors}, {max_neighbors}]")
    print(f"    Spectra with <{k} different-molecule neighbors: {sum(1 for n in neighbors_kept_per_spectrum if n < k):,}")

print(f"\n{'='*80}")
print(f"✓ Exclusive versions created")

## 4. Cache All kNN Graphs

In [ ]:
print("Caching kNN graphs...")
print("="*80)

cache_info = []

for k in k_values:
    # Inclusive
    cache_path = results_dir / f'knn_k{k}_inclusive.pkl'
    joblib.dump(knn_graphs_inclusive[k], cache_path)
    file_size = cache_path.stat().st_size / 1024 / 1024
    print(f"✅ {cache_path.name:35} ({file_size:6.2f} MB)")
    cache_info.append((f'knn_k{k}_inclusive', file_size))
    
    # Exclusive
    cache_path = results_dir / f'knn_k{k}_exclusive.pkl'
    joblib.dump(knn_graphs_exclusive[k], cache_path)
    file_size = cache_path.stat().st_size / 1024 / 1024
    print(f"✅ {cache_path.name:35} ({file_size:6.2f} MB)")
    cache_info.append((f'knn_k{k}_exclusive', file_size))

print(f"\n{'='*80}")
print(f"✓ All {len(cache_info)} graphs cached")

## 5. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("SUMMARY: Nearest-Neighbor Graph Statistics")
print("="*80)

for k in k_values:
    print(f"\nk = {k}")
    
    # Inclusive
    sims_incl = knn_graphs_inclusive[k]['similarities']
    print(f"\n  INCLUSIVE (all neighbors):")
    print(f"    Mean similarity: {np.nanmean(sims_incl):.4f}")
    print(f"    Std similarity:  {np.nanstd(sims_incl):.4f}")
    print(f"    Min similarity:  {np.nanmin(sims_incl):.4f}")
    print(f"    Max similarity:  {np.nanmax(sims_incl):.4f}")
    
    # Exclusive
    sims_excl = knn_graphs_exclusive[k]['similarities']
    # Filter out NaN values (padded entries)
    sims_excl_valid = sims_excl[~np.isnan(sims_excl)]
    print(f"\n  EXCLUSIVE (different-molecule only):")
    print(f"    Mean similarity: {np.nanmean(sims_excl):.4f}")
    print(f"    Std similarity:  {np.nanstd(sims_excl):.4f}")
    if len(sims_excl_valid) > 0:
        print(f"    Min similarity:  {np.nanmin(sims_excl):.4f}")
        print(f"    Max similarity:  {np.nanmax(sims_excl):.4f}")
        print(f"    Valid neighbors (non-NaN): {len(sims_excl_valid)} / {sims_excl.size}")

print(f"\n{'='*80}")

## 6. Write Statistics Log

In [ ]:
# Write detailed stats to file
stats_path = results_dir / 'knn_graphs_stats.txt'

with open(stats_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TASK 1 — Nearest-Neighbor Graph Statistics\n")
    f.write("="*80 + "\n\n")
    
    f.write("## Dataset\n")
    f.write(f"- Total spectra: {embeddings.shape[0]:,}\n")
    f.write(f"- Embedding dimension: {embeddings.shape[1]}\n")
    f.write(f"- Unique molecules: {len(molecule_to_spectra):,}\n")
    f.write(f"- Distance metric: cosine\n")
    f.write(f"\n")
    
    f.write("## Cached Files\n")
    for cache_name, file_size in cache_info:
        f.write(f"- {cache_name}.pkl ({file_size:.2f} MB)\n")
    f.write(f"\n")
    
    for k in k_values:
        f.write(f"## k = {k}\n\n")
        
        sims_incl = knn_graphs_inclusive[k]['similarities']
        sims_excl = knn_graphs_exclusive[k]['similarities']
        
        f.write(f"### INCLUSIVE (all k neighbors)\n")
        f.write(f"- Mean similarity: {np.nanmean(sims_incl):.6f}\n")
        f.write(f"- Std similarity:  {np.nanstd(sims_incl):.6f}\n")
        f.write(f"- Min similarity:  {np.nanmin(sims_incl):.6f}\n")
        f.write(f"- Max similarity:  {np.nanmax(sims_incl):.6f}\n")
        f.write(f"\n")
        
        f.write(f"### EXCLUSIVE (different-molecule only)\n")
        sims_excl_valid = sims_excl[~np.isnan(sims_excl)]
        if len(sims_excl_valid) > 0:
            f.write(f"- Mean similarity: {np.nanmean(sims_excl):.6f}\n")
            f.write(f"- Std similarity:  {np.nanstd(sims_excl):.6f}\n")
            f.write(f"- Min similarity:  {np.nanmin(sims_excl):.6f}\n")
            f.write(f"- Max similarity:  {np.nanmax(sims_excl):.6f}\n")
            f.write(f"- Valid neighbors: {len(sims_excl_valid)} / {sims_excl.size} ({100*len(sims_excl_valid)/sims_excl.size:.1f}%)\n")
        f.write(f"\n")
    
    f.write(f"{'='*80}\n")
    f.write(f"TASK 1 COMPLETE\n")
    f.write(f"{'='*80}\n")

print(f"✅ Statistics saved to: {stats_path}")

## 7. Display Statistics

In [ ]:
# Read and display stats
with open(stats_path, 'r') as f:
    print(f.read())

## 8. Task Complete

In [ ]:
print("\n" + "="*80)
print("TASK 1 — kNN Graph Construction COMPLETE")
print("="*80)
print(f"\nCached files ready for downstream analysis:")
for k in k_values:
    print(f"  - knn_k{k}_inclusive.pkl")
    print(f"  - knn_k{k}_exclusive.pkl")
print(f"\nNext task:")
print(f"  - TASK 2: Indicator 1 (Nearest-Neighbor Descriptor Consistency)")
print(f"  - TASK 3: Indicator 2 (Clustering Purity)")
print(f"  - TASK 4: Indicator 3 (Structural Separation)")
print("="*80)